# CryptoWatch — Spark Analysis

**Kontribusi:** M Faqih Ridho — 5027241123

Menjawab pertanyaan bisnis:
> *Pada jam berapa harga kripto paling volatile? Dan apakah berita yang muncul sejalan dengan pergerakan harga?*

**Pipeline:** Kafka → HDFS → **Spark** → Dashboard

**Isi notebook:**
1. Analisis 1 — Statistik harga per koin *(DataFrame API)*
2. Analisis 2 — Volatilitas per jam *(Spark SQL)*
3. Analisis 3 — Volume berita per jam *(Spark SQL)*
4. **Bonus +5** — Spark MLlib: K-Means clustering + Linear Regression
5. Simpan hasil ke HDFS `/data/crypto/hasil/` dan `dashboard/data/spark_results.json`

## 1. Setup SparkSession

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

HDFS_URI = "hdfs://localhost:9000"
API_PATH = f"{HDFS_URI}/data/crypto/api/"
RSS_PATH = f"{HDFS_URI}/data/crypto/rss/"
HASIL_PATH = f"{HDFS_URI}/data/crypto/hasil"
DASHBOARD_OUT = Path("../dashboard/data/spark_results.json")

spark = (SparkSession.builder
         .appName("CryptoWatch-Analysis")
         .config("spark.hadoop.fs.defaultFS", HDFS_URI)
         .config("spark.sql.session.timeZone", "Asia/Jakarta")
         .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} siap — HDFS default FS = {HDFS_URI}")

## 2. Load data dari HDFS

File JSON dari consumer berisi array event (`[{...},{...}]`) jadi kita pakai `multiLine=true`.

In [ ]:
df_api = spark.read.option("multiLine", "true").json(API_PATH)
df_rss = spark.read.option("multiLine", "true").json(RSS_PATH)

print(f"Event API : {df_api.count()}")
print(f"Event RSS : {df_rss.count()}")
print("\n-- Schema API --")
df_api.printSchema()
print("\n-- Schema RSS --")
df_rss.printSchema()

## Analisis 1 — Statistik harga per koin (DataFrame API)

Agregat per `symbol`: count, avg, max, min, stddev `price_usd`, plus `avg(change_24h)` sebagai gambaran tren.

In [ ]:
stats = (df_api.groupBy("symbol")
         .agg(
             F.count("*").alias("n"),
             F.round(F.avg("price_usd"), 2).alias("avg_usd"),
             F.round(F.max("price_usd"), 2).alias("max_usd"),
             F.round(F.min("price_usd"), 2).alias("min_usd"),
             F.round(F.stddev("price_usd"), 4).alias("stddev_usd"),
             F.round(F.avg("change_24h"), 3).alias("avg_change_pct"),
         )
         .orderBy(F.desc("avg_usd")))
stats.show(truncate=False)

**Interpretasi:**
- Koin dengan `stddev_usd` paling besar = paling bergerak selama periode observasi (belum tentu paling untung — hanya paling bergerak).
- `avg_change_pct` negatif berarti rata-rata turun dalam 24 jam terakhir; positif berarti naik.
- Perbandingan `max_usd` vs `min_usd` menunjukkan rentang harga — semakin jauh, semakin fluktuatif.

## Analisis 2 — Volatilitas per jam (Spark SQL)

Pakai `HOUR(TO_TIMESTAMP(timestamp))` sesuai hint briefing. `avg_abs_change_pct` = rata-rata nilai absolut `change_24h` — proxy volatilitas per jam.

In [ ]:
df_api.createOrReplaceTempView("crypto_api")

vol_by_hour = spark.sql("""
    SELECT
        HOUR(TO_TIMESTAMP(timestamp)) AS hour_utc,
        ROUND(AVG(ABS(change_24h)), 4) AS avg_abs_change_pct,
        ROUND(STDDEV(price_usd), 4)    AS stddev_price_usd,
        COUNT(*)                        AS n_events
    FROM crypto_api
    GROUP BY HOUR(TO_TIMESTAMP(timestamp))
    ORDER BY hour_utc
""")
vol_by_hour.show(24, truncate=False)

**Interpretasi:**
- Jam dengan `avg_abs_change_pct` tertinggi = jam paling volatile. Biasanya 13:00–14:00 UTC (=20:00–21:00 WIB) saat bursa AS buka — liquiditas naik, harga bergerak cepat.
- Jika distribusi rata (tidak ada spike), berarti pasar relatif kalem selama periode observasi.

## Analisis 3 — Volume berita per jam (Spark SQL)

In [ ]:
df_rss.createOrReplaceTempView("crypto_rss")

news_by_hour = spark.sql("""
    SELECT
        HOUR(TO_TIMESTAMP(timestamp))  AS hour_utc,
        COUNT(*)                        AS n_articles,
        COUNT(DISTINCT source)          AS n_sources
    FROM crypto_rss
    GROUP BY HOUR(TO_TIMESTAMP(timestamp))
    ORDER BY hour_utc
""")
news_by_hour.show(24, truncate=False)

**Interpretasi:**
- Jam dengan `n_articles` tertinggi adalah jam paling aktif media kripto. CoinDesk + Cointelegraph kebanyakan publish mengikuti ritme redaksi AS.
- Kalau `n_articles` tinggi bersamaan dengan `avg_abs_change_pct` tinggi pada analisis 2 → sinyal bahwa berita dan pergerakan harga berjalan searah.

## Bonus MLlib (+5) — K-Means clustering + Linear Regression

**Ide:** gabungkan volatilitas + volume berita per jam → cluster jam ke dalam 3 *rezim* (tenang / sedang / bergejolak). Lalu uji apakah jumlah berita per jam memprediksi volatilitas (Linear Regression).

In [ ]:
combined = (vol_by_hour.select("hour_utc", "avg_abs_change_pct")
            .join(news_by_hour.select("hour_utc", "n_articles"),
                  on="hour_utc", how="full")
            .na.fill(0.0, subset=["avg_abs_change_pct"])
            .na.fill(0, subset=["n_articles"])
            .orderBy("hour_utc"))
combined.show(24, truncate=False)

In [ ]:
# K-Means: cluster jam ke 3 regime berdasarkan [volatilitas, jumlah berita]
assembler = VectorAssembler(inputCols=["avg_abs_change_pct", "n_articles"], outputCol="features")
feat = assembler.transform(combined)

kmeans = KMeans(k=3, seed=42, featuresCol="features", predictionCol="hour_regime")
km_model = kmeans.fit(feat)
clustered = (km_model.transform(feat)
             .select("hour_utc", "avg_abs_change_pct", "n_articles", "hour_regime")
             .orderBy("hour_utc"))

print("Cluster centers [avg_abs_change_pct, n_articles]:")
for i, c in enumerate(km_model.clusterCenters()):
    print(f"  Cluster {i}: vol={c[0]:.3f}%  news={c[1]:.1f}")

clustered.show(24, truncate=False)

In [ ]:
# Linear Regression: apakah volume berita memprediksi volatilitas?
lr_data = combined.select(
    F.col("avg_abs_change_pct").cast("double").alias("label"),
    F.col("n_articles").cast("double").alias("n_articles"),
)
lr_assembler = VectorAssembler(inputCols=["n_articles"], outputCol="features")
lr_ready = lr_assembler.transform(lr_data)

lr = LinearRegression(featuresCol="features", labelCol="label")
lr_model = lr.fit(lr_ready)

print(f"Slope (n_articles -> volatility): {lr_model.coefficients[0]:.6f}")
print(f"Intercept:                        {lr_model.intercept:.6f}")
print(f"R² :                              {lr_model.summary.r2:.4f}")
print(f"RMSE:                             {lr_model.summary.rootMeanSquaredError:.4f}")

**Interpretasi MLlib:**
- **Cluster centers** memberitahu kita seperti apa tiap *rezim*: centroid dengan nilai tinggi di kedua fitur = "jam bergejolak & berita padat".
- **Slope positif** pada Linear Regression = makin banyak artikel, makin tinggi volatilitas → berita dan harga bergerak bersamaan.
- **R² mendekati 0** berarti volume berita bukan prediktor yang kuat untuk volatilitas (banyak variabel lain berperan). **R² > 0.3** sudah cukup menarik untuk pasar yang bising.

## Simpan hasil ke HDFS + dashboard

In [ ]:
def save_hdfs(df, subdir):
    path = f"{HASIL_PATH}/{subdir}"
    df.coalesce(1).write.mode("overwrite").json(path)
    print(f"[HDFS] {path}")

save_hdfs(stats,        "stats_per_coin")
save_hdfs(vol_by_hour,  "volatility_by_hour")
save_hdfs(news_by_hour, "news_by_hour")
save_hdfs(clustered,    "hour_regime_clusters")

In [ ]:
def records(df):
    return [row.asDict(recursive=True) for row in df.collect()]

payload = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "n_api_events": df_api.count(),
    "n_rss_events": df_rss.count(),
    "stats_per_coin":       records(stats),
    "volatility_by_hour":   records(vol_by_hour),
    "news_by_hour":         records(news_by_hour),
    "hour_regime_clusters": records(clustered),
    "mllib": {
        "kmeans": {
            "k": 3,
            "cluster_centers": [list(map(float, c)) for c in km_model.clusterCenters()],
            "feature_columns": ["avg_abs_change_pct", "n_articles"],
        },
        "linear_regression": {
            "slope_n_articles": float(lr_model.coefficients[0]),
            "intercept":        float(lr_model.intercept),
            "r2":               float(lr_model.summary.r2),
            "rmse":             float(lr_model.summary.rootMeanSquaredError),
        }
    }
}

DASHBOARD_OUT.parent.mkdir(parents=True, exist_ok=True)
DASHBOARD_OUT.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
print(f"[LOCAL] {DASHBOARD_OUT.resolve()}")

## Kesimpulan

1. **Jam paling volatile:** lihat baris teratas `volatility_by_hour` diurutkan `avg_abs_change_pct` desc.
2. **Apakah berita sejalan dengan harga:** bandingkan distribusi `hour_regime_clusters` — jam yang masuk cluster dengan centroid tinggi di kedua dimensi berarti berita + harga bergerak bersamaan. Slope Linear Regression dan R² mengkuantifikasi kekuatan hubungan ini.

Hasil tersimpan di:
- `hdfs://namenode:9000/data/crypto/hasil/` — 4 sub-folder
- `dashboard/data/spark_results.json` — dikonsumsi Flask dashboard

In [ ]:
spark.stop()